[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Placeholders and Identifiers &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with the `events` table and an empty `tags`. Run it first.
Task 3 writes to `tags`, so run it before anything that reads that table.


In [1]:
import getpass
import os
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from psycopg import errors, sql

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

def build_tags():
    """A small table with a unique column, for the writing half of this notebook."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        conn.execute("DROP TABLE IF EXISTS tags")
        conn.execute("CREATE TABLE tags (id serial PRIMARY KEY, name text UNIQUE, weight int)")


def tags():
    """Everything in tags, as a list of pairs."""
    with psycopg.connect("dbname=guide") as conn:
        return conn.execute("SELECT name, weight FROM tags ORDER BY id").fetchall()


print("server:", start_server())
print(report())
build_tags()
print("tags is ready:", tags())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows
tags is ready: []


**1.** A value that is made of SQL.


In [2]:
awkward = "O'Brien'); DROP TABLE tags; --"

with psycopg.connect("dbname=guide") as conn:
    came_back = conn.execute("SELECT %s AS who", (awkward,)).fetchone()[0]

print("sent:      ", awkward)
print("came back: ", came_back)
print("unchanged: ", came_back == awkward)
print("tags still exists:", tags() is not None)


sent:       O'Brien'); DROP TABLE tags; --
came back:  O'Brien'); DROP TABLE tags; --
unchanged:  True
tags still exists: True


The apostrophes, the parenthesis, the semicolon and the comment marker all came back as characters.
None of them was ever part of a statement, so none of them could be syntax.


**2.** A list of ids.


In [3]:
wanted = [3, 14, 15, 92, 65]

with psycopg.connect("dbname=guide") as conn:
    found = conn.execute(
        "SELECT count(*) FROM events WHERE id = ANY(%s)", (wanted,)).fetchone()[0]
    rows = conn.execute(
        "SELECT id, kind FROM events WHERE id = ANY(%s) ORDER BY id", (wanted,)).fetchall()

print("asked for", len(wanted), "ids and found", found)
print(rows)


asked for 5 ids and found 5
[(3, 'click'), (14, 'purchase'), (15, 'click'), (65, 'purchase'), (92, 'purchase')]


One value goes over, an array, and the statement text is the same whether the list has five items or
five hundred. `IN %s` would have raised, because `IN` wants a parenthesized list written into the
statement and a list is a value.


**3.** Three rows, then two with their ids.


In [4]:
build_tags()

with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    cur.executemany("INSERT INTO tags (name, weight) VALUES (%s, %s)",
                    [("north", 1), ("south", 2), ("east", 3)])
    print("first call wrote:", cur.rowcount, "rows")

with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    cur.executemany("INSERT INTO tags (name, weight) VALUES (%s, %s) RETURNING id",
                    [("west", 4), ("centre", 5)], returning=True)
    ids = []
    while True:
        ids.append(cur.fetchone()[0])
        if not cur.nextset():
            break

print("second call gave ids:", ids)
print("rows now:", tags())


first call wrote: 3 rows
second call gave ids: [4, 5]
rows now: [('north', 1), ('south', 2), ('east', 3), ('west', 4), ('centre', 5)]


Without `returning=True` there is nothing to fetch, because psycopg does not keep results it was not
asked for. With it, each set of values leaves its own result behind, which is what `nextset` walks.


**4.** A table name in a variable.


In [5]:
table = "tags"

with psycopg.connect("dbname=guide") as conn:
    query = sql.SQL("SELECT count(*) FROM {}").format(sql.Identifier(table))
    print("the statement:", query.as_string(conn))
    print("the answer:   ", conn.execute(query).fetchone())


the statement: SELECT count(*) FROM "tags"
the answer:    (5,)


The name went into the statement, quoted as a name, before anything was sent. That is the difference
from a value: this one had to be part of the text, because the server has to know which table it is
reading before it can plan the query at all.


**5.** A name with a quote in it.


In [6]:
with psycopg.connect("dbname=guide") as conn:
    for name in ('plain', 'has "quotes"', 'has ""two"" already'):
        print(f"  {name!r:<24} -> {sql.Identifier(name).as_string(conn)}")


  'plain'                  -> "plain"
  'has "quotes"'           -> "has ""quotes"""
  'has ""two"" already'    -> "has """"two"""" already"


Each embedded double quote is doubled, which is how PostgreSQL escapes a quote inside a quoted name.
Doing that by hand is exactly the kind of thing that is right until it is not, which is why
`Identifier` exists.


**6.** asyncpg, and a value used twice.


In [7]:
conn = await asyncpg.connect(database="guide")

once = await conn.fetch("SELECT id, kind FROM events WHERE id = $1", 7)
twice = await conn.fetch(
    "SELECT id, kind FROM events WHERE id = $1 OR id = $1 + 1 ORDER BY id", 7)

print("one place: ", [dict(r) for r in once])
print("two places:", [dict(r) for r in twice])
await conn.close()


one place:  [{'id': 7, 'kind': 'view'}]
two places: [{'id': 7, 'kind': 'view'}, {'id': 8, 'kind': 'purchase'}]


`$1` appears twice in the second query and the value was passed once, which numbered placeholders
allow and `%s` does not: psycopg would need the value given twice, because each `%s` consumes one.


---

&#8592; **Back to:** [Placeholders and Identifiers](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/04-placeholders-and-identifiers.ipynb)  &nbsp;&middot;&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
